# AutoMorphClass on three shapes, biomarker by biomarker

A close reading of one implementation. The
[main notebook](../biomarker-synthetic.ipynb) compares seven programs across nine shapes and answers
*which of these agree*; this one asks a narrower question about
[AutoMorphClass](../../../docs/projects/automorphclass.md) alone — **what does it actually return, on the three
simplest shapes, at each angle, beside what the geometry requires?**

Everything is under **AutoMorphClass's own column names**, not the catalogued ones. That is the point:
the catalogued names exist to make two programs comparable, and there is only one program here, so
the names its authors chose are the ones a reader checking against their documentation will look
for. Where a column maps onto a catalogued biomarker the mapping is shown beside it, and where the
shape settles a theoretical value that is shown too.

**The three shapes, in order of how much they ask for:**

| Shape | What it is | Why it comes first |
| --- | --- | --- |
| `straight` | one straight vessel per class, leaving the optic disc | every curvature measure must read zero and every tortuosity exactly 1 |
| `arc` | a circular arc per class | curvature is exactly 1/r, and the arc-to-chord ratio has a closed form |
| `sinusoid` | a sine wave per class | the only one of the three whose curvature changes sign |

A blank in the theory column means **this shape settles no value for that quantity** — which is not
the same as the quantity being wrong, and not the same as AutoMorph declining to answer it.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display


def repository() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").is_dir():
            return candidate
    raise RuntimeError(f"nothing above {Path.cwd()} looks like the fundus-atlas repository")


ROOT = repository()
sys.path.insert(0, str(ROOT / "src"))

from benchmarks import biomarker_synthetic as benchmark  # noqa: E402
from benchmarks.shapes import store  # noqa: E402
from biomarkers import canonical  # noqa: E402
from biomarkers.utils import catalogue  # noqa: E402

#: The one implementation this notebook reads. Change it and everything below follows.
IMPLEMENTATION = "automorphclass"

#: In order of how much they ask for: a straight vessel, then a curve, then a curve that changes
#: its mind about which way it bends.
SHAPES = ("straight", "arc", "sinusoid")

EVIDENCE = ROOT / "results" / benchmark.NAME / IMPLEMENTATION
STORE = ROOT / store.STORE
pd.set_option("display.width", 200, "display.max_rows", 200)

#: AutoMorph's own column name → the catalogued biomarker it answers to, or None where the
#: catalogue has no name for it. This is the adapter's claim, not the run's.
NAMES = catalogue.load(IMPLEMENTATION).declare()["names"]

TRUTH = pd.read_csv(STORE / "ground_truth.csv").set_index("key")

#: How many microns a pixel is worth, read from the manifest rather than assumed.
UM_PER_PX = float(pd.DataFrame(store.read(STORE))["um_per_px"].iloc[0])
print(f"{IMPLEMENTATION}: {len(NAMES)} columns, "
      f"{sum(1 for v in NAMES.values() if v)} of them with a catalogued name")

automorphclass: 24 columns, 17 of them with a catalogued name


In [2]:
def readings(shape: str) -> pd.DataFrame:
    """What AutoMorph returned for one shape: a row per column, a column per angle.

    The theoretical value is joined on last and only where the shape settles one, so a blank there
    **Every number in a row is in the implementation's own units, which are pixels.** A catalogued
    biomarker is in microns, so the theory is converted *back* onto the pixel grid before it is put
    beside figures this program actually printed. The `unit` column says what the catalogued name
    is in.

    The theoretical value is joined on last, so a blank is a statement about the fixture rather
    than about AutoMorph.
    """
    measured = pd.read_csv(EVIDENCE / f"{shape}.csv")
    rows = {}
    for own in NAMES:
        values = {}
        for _, rendering in measured.iterrows():
            values[f"{rendering['rotation']:.0f}°"] = rendering.get(f"said_{own}")
        catalogued = NAMES[own]
        truth, unit = np.nan, ""
        if catalogued:
            unit = canonical.unit(catalogued)
            keys = [k for k in TRUTH.index if k.startswith(f"{shape}-")]
            if keys and catalogued in TRUTH.columns:
                # Back onto the pixel grid: the inverse of the one conversion this repository makes.
                truth = canonical.from_pixels(
                    catalogued, TRUTH.loc[keys[0], catalogued], 1.0 / UM_PER_PX
                )
        rows[own] = {
            **values,
            "theory, in px": truth,
            "catalogued as": catalogued or "—",
            "unit": unit or "—",
        }
    frame = pd.DataFrame(rows).T
    frame.index.name = f"{IMPLEMENTATION}'s own name"
    return frame


def show(shape: str) -> pd.DataFrame:
    frame = readings(shape)
    angles = [c for c in frame.columns if c.endswith("°")]
    numeric = frame[angles + ["theory, in px"]].astype(float)
    spread = numeric[angles].max(axis=1) - numeric[angles].min(axis=1)
    out = numeric.round(4)
    out["moves"] = spread.round(4)
    out["catalogued as"] = frame["catalogued as"]
    out["unit"] = frame["unit"]
    return out

## 1. `straight`

A single straight vessel per class, leaving the optic disc. **Everything about it is known
exactly**: the arc-to-chord ratio is 1, every curvature measure is 0, and the width is the width it
was drawn at. Nothing here is a hard measurement, which is what makes it the right place to start —
a value that is wrong on a straight line is wrong for a reason that has nothing to do with
difficulty.

In [3]:
show("straight")

,0°,15°,30°,45°,60°,75°,90°,"theory, in px",moves,catalogued as,unit
automorphclass's own name,,,,,,,,,,,
vessel_density_vessels,0.0048,0.0049,0.0049,0.0048,0.0049,0.0049,0.0048,0.0049,0.0001,density/over-image/vessels,1
fractal_dimension_vessels,0.9702,0.9842,1.0126,0.9777,1.0027,0.9916,0.9702,1.0000,0.0423,density/box-counting/vessels,1
average_width_vessels,15.8518,16.6384,18.5825,22.3951,18.6164,16.6937,15.9022,15.8025,6.5433,calibre/width/vessels/length-weighted,µm
distance_tortuosity_vessels,1.0013,1.0738,1.0732,1.0000,1.0732,1.0733,1.0023,1.0000,0.0738,tortuosity/hart-tau1/vessels/length-weighted,1
squared_curvature_tortuosity_vessels,0.4349,25.5045,5.6133,0.0000,5.5137,25.3241,0.8508,NaN,25.5045,—,—
tortuosity_density_vessels,0.0001,0.0192,0.0073,0.0000,0.0099,0.0140,0.0003,0.0000,0.0192,tortuosity/grisan-density/vessels/length-weighted,1/µm
vessel_density_artery,0.0018,0.0019,0.0019,0.0018,0.0019,0.0019,0.0018,0.0019,0.0001,density/over-image/artery,1
fractal_dimension_artery,0.8646,0.8639,0.8855,0.8243,0.8855,0.8639,0.8646,1.0000,0.0612,density/box-counting/artery,1
average_width_artery,12.2413,13.2349,14.7832,17.2299,14.8645,13.2782,12.2803,12.6420,4.9886,calibre/width/artery/length-weighted,µm


## 2. `arc`

A circular arc per class, the two sharing a subtended angle and differing in radius. The
arc-to-chord ratio is `θ / (2 sin(θ/2))` in closed form and **independent of radius**, so both
classes must return the same tortuosity while their curvatures differ. That pairing is the test:
an implementation that returns different tortuosities for the two is measuring the radius somewhere
it should not.

In [4]:
show("arc")

,0°,15°,30°,45°,60°,75°,90°,"theory, in px",moves,catalogued as,unit
automorphclass's own name,,,,,,,,,,,
vessel_density_vessels,0.0050,0.0050,0.0050,0.0050,0.0050,0.0050,0.0050,0.0050,0.0000,density/over-image/vessels,1
fractal_dimension_vessels,1.0608,1.1006,1.0979,1.0351,1.0420,1.0669,1.0608,1.0000,0.0655,density/box-counting/vessels,1
average_width_vessels,17.8172,17.3959,16.6714,16.8635,17.5819,17.9542,17.8623,15.3510,1.2828,calibre/width/vessels/length-weighted,µm
distance_tortuosity_vessels,1.1241,1.1168,1.1285,1.1240,1.1163,1.1282,1.1243,1.0650,0.0122,tortuosity/hart-tau1/vessels/length-weighted,1
squared_curvature_tortuosity_vessels,9.6389,11.5993,12.7999,12.6124,10.5658,10.4154,9.6523,NaN,3.1610,—,—
tortuosity_density_vessels,0.0081,0.0084,0.0099,0.0090,0.0092,0.0094,0.0083,0.0000,0.0018,tortuosity/grisan-density/vessels/length-weighted,1/µm
vessel_density_artery,0.0024,0.0024,0.0024,0.0024,0.0024,0.0024,0.0024,0.0024,0.0000,density/over-image/artery,1
fractal_dimension_artery,0.9445,0.9871,0.9886,0.9248,0.9317,0.9530,0.9445,1.0000,0.0638,density/box-counting/artery,1
average_width_artery,14.5767,14.2248,13.6386,13.7723,14.4155,14.6677,14.6415,12.6420,1.0290,calibre/width/artery/length-weighted,µm


## 3. `sinusoid`

A sine wave per class, and **the only one of the three whose curvature changes sign**. Its arc
length is an elliptic integral with no elementary closed form, so the theoretical value here is the
integral rather than a formula. It is the shape that settles the inflection count and Grisan's
density, both of which are zero on everything else.

In [5]:
show("sinusoid")

,0°,15°,30°,45°,60°,75°,90°,"theory, in px",moves,catalogued as,unit
automorphclass's own name,,,,,,,,,,,
vessel_density_vessels,0.0055,0.0055,0.0055,0.0055,0.0055,0.0055,0.0055,0.0055,0.0000,density/over-image/vessels,1
fractal_dimension_vessels,1.0231,1.0355,1.0376,1.0355,1.0434,1.0569,1.0231,1.0000,0.0338,density/box-counting/vessels,1
average_width_vessels,18.3603,18.2091,17.7653,17.4402,17.4774,17.9443,18.3748,15.8025,0.9346,calibre/width/vessels/length-weighted,µm
distance_tortuosity_vessels,1.2025,1.2014,1.2002,1.2019,1.2014,1.2019,1.2024,1.1398,0.0023,tortuosity/hart-tau1/vessels/length-weighted,1
squared_curvature_tortuosity_vessels,9.9499,11.1218,11.9953,13.0723,12.5482,12.5478,9.8964,NaN,3.1759,—,—
tortuosity_density_vessels,0.0091,0.0089,0.0093,0.0083,0.0084,0.0097,0.0087,0.0006,0.0014,tortuosity/grisan-density/vessels/length-weighted,1/µm
vessel_density_artery,0.0022,0.0022,0.0022,0.0022,0.0022,0.0022,0.0022,0.0022,0.0000,density/over-image/artery,1
fractal_dimension_artery,0.8920,0.8980,0.9306,0.9174,0.9376,0.9395,0.8920,1.0000,0.0475,density/box-counting/artery,1
average_width_artery,14.5710,14.4094,13.9381,13.8441,13.9833,14.4476,14.5940,12.6420,0.7499,calibre/width/artery/length-weighted,µm


## 4. What the three tables show

AutoMorphClass is a rewrite of [AutoMorph](biomarker-synthetic-automorph.ipynb)'s measuring stage,
and the most useful way to read these tables is against that one.

### 4.1 Every reading is a possible value, which is not true of its ancestor

AutoMorph returns **exactly 0.0000** for `distance_tortuosity` on `straight` at 30° and 60°, and on
`sinusoid` at 0° and 90°. An arc over a chord cannot be less than 1, so those are failures reported
as numbers. **Nothing in these three tables is impossible**: every tortuosity is at or above 1,
every width is positive, every fractal dimension is between 0 and 2. Whatever else follows, the
rewrite fixed the thing that most needed fixing, because a wrong number you can see is worth more
than a failure you cannot.

### 4.2 Steady, and biased high

On `arc`, where the geometry requires **1.1107** at every angle and for both classes:

| | 0° | 30° | 60° | 90° | theory | moves |
| --- | --- | --- | --- | --- | --- | --- |
| `distance_tortuosity_artery` | 1.1697 | 1.1704 | 1.1715 | 1.1739 | 1.1107 | 0.0042 |
| `distance_tortuosity_vein` | 1.1703 | 1.1722 | 1.1723 | 1.1704 | 1.1107 | 0.0020 |

Turning the picture moves it by **0.4%** — and it is **5.3% too high at every angle**. The two
classes agree with each other to three decimals, which is what the geometry requires of them, and
both agree on the wrong value. That is the opposite failure from AutoMorph, whose vein reads 1.037,
1.062, 7.133 and 5.367 on the same shape: a steady bias can be corrected once it is known, and
noise of that size cannot be corrected at all.

The same holds on `sinusoid` — 1.43 to 1.46 against a required 1.3726, about 5% high and moving by
2% — and on `straight`, where it reads 1.0007 at the axis-aligned angles and 1.073 at the diagonal
ones.

### 4.3 The width error runs the opposite way to AutoMorph's

On `straight`, one unchanging vessel at four angles:

| | 0° | 30° | 60° | 90° | drawn at |
| --- | --- | --- | --- | --- | --- |
| `average_width_vein`, **AutoMorphClass** | 24.87 | 28.52 | 28.41 | 24.87 | 24 |
| `average_width_vein`, **AutoMorph** | 14.15 | 24.94 | 25.13 | 14.15 | 24 |

Both wobble with the angle and they wobble **in opposite directions**. AutoMorphClass is nearly
exact where the vessel is axis-aligned and 18% wide where it is diagonal; AutoMorph is nearly exact
where it is diagonal and 41% narrow where it is axis-aligned. One of the two rewrote the width
calculation and the tables cannot say which is nearer right in general — only that on this shape
the rewrite is closer, and that the two cannot both be measuring what they claim to measure.

### 4.4 A straight line with a fractal dimension of 0.376

`fractal_dimension_artery` on `straight` reads **0.3758, 0.8849, 0.8849, 0.3758**. A straight line
has dimension 1, and a bounded raster of one will read a little under it — 0.88 is a plausible
underestimate and 0.376 is not a dimension of anything.

**AutoMorph does not do this.** Its `fractal_dimension_artery` on the same shape reads 0.927,
0.977, 0.977, 0.927. So this is something the rewrite introduced rather than something it
inherited, and it appears only at the two axis-aligned angles.

### 4.5 What is not checked here

Every fractal dimension, both per-class vessel densities and `squared_curvature_tortuosity` carry
no theory on these shapes — the first two because these shapes settle no value for them, the last
because the catalogue has no name for it at all. Those rows are reported and unchecked, which is a
third state and not a pass.
